# Fixed-budget oracle

Choose exactly $R$ of $N$ rows to maximize retained importance per unit of modeled latency. A selected mask $M$ consists of maximal contiguous runs $\mathcal C(M)$:

$$I(M)=\sum_{i=0}^{N-1}v_iM_i,\qquad
L(M)=\sum_{C\in\mathcal C(M)}T(|C|),\qquad
\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.$$

| Symbol | Meaning |
|---|---|
| $v_i\ge0$ | Importance of row $i$ |
| $N$ | Number of rows in the trace |
| $R$ | Requested number of selected rows, $1\le R\le N$ |
| $T(\ell)$ | Modeled latency of one contiguous run of length $\ell$ |
| $P_j$ | Total importance before row $j$: $\sum_{i<j}v_i$ |

The interface is:

```python
problem = SelectionProblem(v, cost)
mask, lower, upper, calls = problem.solve(R)
```

`mask` selects exactly `R` rows. `lower` is its importance-to-latency ratio, `upper` bounds the optimum, and `calls` counts Bellman solves. The bounds and error guarantees below are mathematical statements in exact arithmetic; the implementation uses float64 and a stopping tolerance.

This experiment reads traces from [02_01](02_01_capture_traces.ipynb) and the fitted latency model from experiment 01. It writes `oracle_reference.csv` for [the selector sweep](02_03_selector_sweep.ipynb) and [analysis](02_04_analyze_oracle.ipynb). All latencies are additive model estimates.

Read in order: [setup](#setup) → [cost model](#cost-model) → [solver](#solver) → [small example](#example) → [experiment inputs](#inputs) → [run and save](#run). [Detailed derivations](#derivations) follow the experiment.

<a id="setup"></a>
## 1. Setup

Run from this directory with NumPy, SciPy, pandas, and tqdm. Configure the trace sample and row budgets here. Input files are read in section 5; the explicit save cell in section 6 replaces `oracle_reference.csv`.

In [22]:
from dataclasses import dataclass
from pathlib import Path

import hashlib
import json

import numpy as np
import pandas as pd
from scipy.ndimage import maximum_filter1d
from tqdm.auto import tqdm

In [23]:
# Input artifacts and output reference.
MODEL = Path("../01/latency_model.json")
TRACE = Path("activation_traces.npz")
RESULTS = Path("oracle_reference.csv")

# Sample this many traces per shape; 0 uses all traces.
MAX_TRACES_PER_SHAPE = 32

# Requested fractions of rows, from 0.1 through 0.9.
BUDGETS = np.arange(1, 10) / 10

<a id="cost-model"></a>
## 2. Cost of a selected run

The two-line model is expressed in row units:

$$T(0)=0,\qquad
T(\ell)=\begin{cases}a+b_1\ell,&1\le\ell\le s,\\b_2\ell,&\ell>s,\end{cases}
\qquad a=(b_2-b_1)s.$$

Here $s\ge0$ is the boundary between short and long runs, and $0\le b_1\le b_2$ with $b_2>0$. The boundary stays real-valued; only the DP's integer window uses $\lfloor s\rfloor$. The intercept $a$ applies to nonempty short runs, while the empty run costs zero. Adjacent selected intervals form one run and are charged once.

`TwoLineCost.table(n)` builds the costs for lengths `0` through `n`. Each `SelectionProblem` builds this table once and reuses it across budgets. Conversion from the fitted model's KiB units to row units is part of [input preparation](#row-units). Lengths outside the measured range use the model's linear extension.

In [24]:
@dataclass(frozen=True)
class TwoLineCost:

    """Two-line latency model in row units."""

    s: float
    b1: float
    b2: float

    def table(self, n):

        """Return the latency for each run length from 0 through n."""

        # [0, 1, .... n]
        lengths = np.arange(n + 1)

        # constant -> [fixed, fixed, .... fixed ]
        fixed_cost = (self.b2 - self.b1) * self.s

        # front    -> [0, 1 ... s, s, s ... ]
        short_part = np.minimum(lengths, self.s)

        # back     -> [0, 0 ... 0, 1, 2 ... ]
        extra_part = np.maximum(lengths - self.s, 0)

        # latency = (constant) + (front) + (back)
        latency = fixed_cost + self.b1 * short_part + self.b2 * extra_part
        latency[0] = 0
        return latency

<a id="solver"></a>
## 3. Solver: ratio search, Bellman, and recovery

For a proposed ratio $q$, the additive subproblem is

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

`solve` uses this subproblem in a Dinkelbach iteration:

1. Initialize `lower` with the best contiguous run. Initialize `upper` with the maximum importance divided by the minimum latency `T[R]`.
2. Set `q = lower` and call `bellman(R, q)` to maximize importance minus `q` times latency.
3. Use the maximum residual `DP[-1, -1]` to tighten `upper`. Recover a maximizing mask and use its actual ratio to improve `lower`.
4. Return the best mask when `upper - lower <= rtol * initial_lower`.

[Bounds and convergence](#bounds) explain why these updates work.

For the Bellman calculation, let $Z=N-R$. `DP[z, r]` is the best additive score over the first $z+r$ rows, with $z$ rows excluded and $r$ selected. With $D_{z,r}$ denoting this array entry,

$$D_{0,r}=P_r-qT(r),\qquad
D_{z,r}=\max_{0\le t\le r}\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].$$

The final selected run is $[z+t,z+r)$, and `DP[Z, R]` equals $F_R(q)$. `recover` follows maximizing choices of $t$ backward. The two-line model lets `bellman` evaluate each row using the `A` and `B` range maxima.

See [the recurrence proof](#recurrence), [the two range maxima](#range-maxima), and [recovery and complexity](#recovery) for the full derivation.

### Array helpers

`runs(mask)` finds the selected intervals, using an exclusive end index. `trailing_max(values, width)` finds the largest value in each trailing window. Both operate on arrays independently of a selection problem.

In [25]:
def runs(mask):

    """Return maximal selected runs as [start, end) intervals.

    Example: [0, 1, 1, 1, 0, 0, 1, 1] gives [[1, 4], [6, 8]].
    """

    # Padding exposes runs touching either boundary.
    padded = np.concatenate(([False], mask, [False]))
    changes = np.diff(padded)
    edges = np.flatnonzero(changes)

    return edges.reshape(-1, 2)

In [26]:
def trailing_max(values, width):

    """Return the maximum of the last width values at each position.

    Example: [-1, -2, -3, -4, -5] with width=3 gives [-1, -1, -1, -2, -3].
    """
    if width < 1:
        raise ValueError("width must be positive")

    # Align the window with [j - width + 1, j]; ignore missing values on the left.
    return maximum_filter1d(
        values,
        size=width,
        origin=(width - 1) // 2,
        mode="constant",
        cval=-np.inf,
    )


### One problem object per trace

`SelectionProblem` owns `v`, `cost`, the prefix sums `P`, and the run-latency table `T`. Create a new object when the values or model change. Each call supplies its row budget `R`; `q`, `DP`, and masks stay local to that calculation.

Read the class from the top: `__init__` prepares the data, `solve` coordinates the search, and `bellman` and `recover` implement its two main steps. Selection and evaluation methods follow. Short local names such as `P`, `T`, `A`, and `B` match the formulas above.

In [27]:
class SelectionProblem:

    """One trace and its cost model, prepared once for all row budgets.

    Create a new problem when the importance values or cost model change.
    R, q, DP, and masks belong to individual calculations.
    """

    def __init__(self, v, cost: TwoLineCost):

        # Own the input values so changes to the caller's array do not affect P.
        self.v     = np.array(v, dtype=float, copy=True)
        self.cost  = cost

        # P[i]: total importance before row i.
        self.P     = np.zeros(len(self.v) + 1)
        self.P[1:] = np.cumsum(self.v)

        # T[length]: latency of one selected run.
        self.T     = cost.table(len(self.v))

    def solve(self, R, rtol=1e-6):

        """Return the best mask, ratio bounds, and number of Bellman calls."""
        T = self.T

        # Lower bound: achievable ratio from the best contiguous run.
        mask = self.best_run(R)
        lower = self.total_importance(mask) / T[R]

        # Upper bound: maximum importance divided by minimum latency.
        upper = self.total_importance(self.top_r(R))    / T[R]

        # Fix the stopping tolerance using the initial lower bound.
        tolerance = rtol * lower
        calls = 0

        while upper - lower > tolerance:

            # Maximize importance - q * latency with q fixed at the current bound.
            q  = lower
            DP = self.bellman(R, q)
            calls += 1

            # DP[-1, -1] is the maximum residual at q.
            # Dividing by the minimum latency T[R] bounds the possible ratio gain.
            residual = max(0.0, DP[-1, -1])
            upper    = min(upper, q + residual / T[R])

            # Recover the maximizing mask and evaluate its actual ratio.
            candidate = self.recover(R, q, DP)
            importance = self.total_importance(candidate)
            latency = self.total_latency(candidate)
            candidate_ratio = importance / latency

            # Keep the best feasible mask and its ratio.
            if candidate_ratio > lower:
                mask = candidate
                lower = candidate_ratio

            # Keep the bounds ordered despite floating-point rounding.
            upper = max(upper, lower)

            # Stop with an error if the bound is still too wide but q cannot improve.
            if lower <= q and upper - lower > tolerance:
                raise RuntimeError("Dinkelbach iteration stalled before meeting rtol.")

        return mask, lower, upper, calls

    def bellman(self, R, q):

        """Compute the best importance - q * latency for each DP state."""

        # Reuse the prepared data; keep the notation used in the recurrence.
        P = self.P
        T = self.T
        cost = self.cost
        s = cost.s
        b1 = cost.b1
        b2 = cost.b2

        # Intercept of the short-run cost; T[0] is handled separately.
        a = (b2 - b1) * s

        # Maximum length of a short run, capped at R.
        h = min(int(np.floor(s)), R)

        # Number of unselected rows.
        Z = len(self.v) - R

        # Candidate t and state r both range from 0 through R.
        t = np.arange(R + 1)
        r = np.arange(R + 1)

        # DP[z, r]: best score over the first z + r rows,
        # with z rows excluded and r rows selected.
        DP = np.empty((Z + 1, R + 1))
        DP[0] = P[:R + 1] - q * T[:R + 1]

        for z in range(1, Z + 1):

            # A[t] = A_{z,t}, B[t] = B_{z,t}.
            A = DP[z - 1, t] - P[z + t] + q * b1 * t
            B = DP[z - 1, t] - P[z + t] + q * b2 * t

            # short_max[r] = max A[t] over max(0, r-h) <= t < r.
            short_max = np.full(R + 1, -np.inf)
            if h:
                short_max[1:] = trailing_max(A[:-1], h)

            # long_max[r] = max B[t] over 0 <= t < r-h.
            long_max = np.full(R + 1, -np.inf)
            long_max[h + 1:] = np.maximum.accumulate(B)[:R - h]

            # Compare empty, short, and long final runs.
            DP[z] = np.maximum.reduce([
                DP[z - 1],
                P[z + r] - q * (a + b1 * r) + short_max,
                P[z + r] - q * b2 * r + long_max,
            ])

        return DP

    def recover(self, R, q, DP):

        """Recover a maximizing mask from the DP table for this R and q."""
        P = self.P
        T = self.T
        Z = len(self.v) - R

        mask = np.zeros(len(self.v), dtype=bool)
        r = R

        for z in range(Z, 0, -1):

            # Candidate selected counts before the last zero.
            t = np.arange(r + 1)
            scores = (
                DP[z - 1, t]           # Best score of the prefix
                + P[z + r] - P[z + t]  # Importance of the final run
                - q * T[r - t]         # Cost penalty of the final run
            )
            best_t = int(np.argmax(scores))

            # Select the final run [z + best_t, z + r).
            mask[z + best_t:z + r] = True

            # Continue from the preceding state (z - 1, best_t).
            r = best_t

        # With no zeros left, select every row in the remaining prefix.
        mask[:r] = True
        return mask

    def best_run(self, R):

        """Select the contiguous run of R rows with maximum importance."""
        P = self.P

        # Compare all contiguous runs of length R.
        run_importance = P[R:] - P[:-R]
        start = int(np.argmax(run_importance))

        mask = np.zeros(len(self.v), dtype=bool)
        mask[start:start + R] = True
        return mask

    def top_r(self, R):

        """Select the R rows with greatest importance."""

        # Sort by decreasing importance; keep the original order for ties.
        order = np.argsort(-self.v, kind="stable")

        mask = np.zeros(len(self.v), dtype=bool)
        mask[order[:R]] = True
        return mask

    def total_importance(self, mask):

        """Return the total importance of selected rows."""
        return self.v[mask].sum()

    def total_latency(self, mask):

        """Return the total latency of selected runs."""
        spans   = runs(mask)
        lengths = spans[:, 1] - spans[:, 0]
        return self.T[lengths].sum()

<a id="example"></a>
## 4. A small example

Select three rows from `[6, 0, 0, 5, 5]`. The best contiguous run has importance 10 and latency 3. Splitting the selection into the first row and the last two rows retains importance 16 at latency 3.8, giving a higher ratio.

The example below shows the selected mask, its measurements, the ratio bracket, and the number of Bellman calls. It uses its own inputs and does not depend on the trace archive.

In [28]:
demo_problem = SelectionProblem(
    np.array([6., 0., 0., 5., 5.]),
    TwoLineCost(s=2., b1=0.2, b2=1.),
)
demo_mask, demo_lower, demo_upper, demo_calls = demo_problem.solve(R=3)

pd.DataFrame([{
    "mask": demo_mask.astype(int).tolist(),
    "importance": demo_problem.total_importance(demo_mask),
    "latency": demo_problem.total_latency(demo_mask),
    "lower": demo_lower,
    "upper": demo_upper,
    "calls": demo_calls,
}])

,mask,importance,latency,lower,upper,calls
0,"[1, 0, 0, 1, 1]",16.0,3.8,4.210526,4.210526,2


<a id="inputs"></a>
## 5. Prepare experiment inputs

Read both input artifacts, select traces, and create one problem per trace. Each problem is reused for all requested budgets. Layers and budgets from the same prompt are related observations.

### Read the traces and fitted model

The archive contains a manifest and one importance vector per trace. The manifest also records the weight dtype needed for conversion to row units.

In [29]:
with np.load(TRACE) as archive:
    manifest_json = archive["manifest_json"].item()
    manifest = json.loads(str(manifest_json))

    traces = []
    for trace_metadata in manifest["traces"]:
        array_key = trace_metadata["array"]
        values = archive[array_key].astype(float)
        traces.append({**trace_metadata, "values": values})

In [30]:
model = json.loads(MODEL.read_text())

# Fitted coefficients from experiment 01_02, in KiB units.
s_kib = model["s_kib"]
c1    = model["c1_ms_per_kib"]
delta = model["delta_ms_per_kib"]

The hashes identify the exact input artifacts. They are saved with each result, and the selector sweep checks them before reusing the bounds.

In [31]:
input_hashes = {
    "trace_sha256": hashlib.sha256(TRACE.read_bytes()).hexdigest(),
    "model_sha256": hashlib.sha256(MODEL.read_bytes()).hexdigest()
}

### Select traces

Keep up to `MAX_TRACES_PER_SHAPE` evenly spaced traces within each shape; zero keeps all traces. Sort the selected traces by `trace_id` so the execution order is deterministic.

In [32]:
if MAX_TRACES_PER_SHAPE:

    # Group traces by shape before sampling.
    traces_by_shape = {}
    for trace in traces:
        shape = trace["shape"]
        if shape not in traces_by_shape:
            traces_by_shape[shape] = []
        traces_by_shape[shape].append(trace)

    sampled_traces = []
    for shape in sorted(traces_by_shape):
        shape_traces = traces_by_shape[shape]

        if len(shape_traces) <= MAX_TRACES_PER_SHAPE:
            sampled_traces.extend(shape_traces)
        else:
            last_index = len(shape_traces) - 1
            indices = np.linspace(0, last_index, MAX_TRACES_PER_SHAPE, dtype=int)
            for index in indices:
                sampled_traces.append(shape_traces[index])

    traces = sampled_traces

traces.sort(key=lambda trace: trace["trace_id"])

<a id="row-units"></a>
### Normalize importance and convert the model to row units

Each nonzero vector keeps all rows and is normalized to mean one; an all-zero vector stays zero. Normalized values are rounded to float32 and stored as float64 for the solver.

For a projection with $N$ inputs and $d$ outputs, one weight row contains $d$ weights. Its size is $w=d\cdot\mathrm{weight\_bytes}/1024$ KiB. The fitted model is

$$T_{\mathrm{KiB}}(z)=c_1z+\delta\max(z_{\mathrm{2L}},z).$$

Substituting $z=w\ell$ gives the row-unit coefficients

$$s=\frac{z_{\mathrm{2L}}}{w},\qquad b_1=wc_1,\qquad b_2=w(c_1+\delta).$$

The JSON field `s_kib` is $z_{\mathrm{2L}}$. Build one `SelectionProblem` per trace so its prepared arrays are shared across budgets.

In [33]:
bytes_per_dtype = {
    "float16": 2,
    "bfloat16": 2,
    "float32": 4,
}
weight_bytes = bytes_per_dtype[manifest["model_dtype"]]

In [34]:
cases = []
for trace in traces:
    v = trace["values"]
    v = (v / v.mean()).astype(np.float32).astype(float) if v.any() else v
    w = weight_bytes * trace["d"] / 1024
    cost    = TwoLineCost(s=s_kib / w, b1=w * c1, b2=w * (c1 + delta))
    problem = SelectionProblem(v, cost)
    cases.append((trace, problem))


<a id="run"></a>
## 6. Run and save the reference

For each trace and requested fraction $f$, select $R=\max(1,\lfloor fN\rfloor)$ rows. Each trace-budget pair is solved once.

### Measurements recorded for each mask

The reporting helper evaluates the returned mask and produces the fields used in the CSV:

| Fields | Interpretation |
|---|---|
| `trace_id`, `shape`, `prompt`, `fraction` | Input identity and requested budget |
| `R_nominal`, `R_selected`, `fill_pct`, `status` | Requested and selected row counts and evaluation status |
| `oracle_importance`, `oracle_latency`, `retained_pct` | Measurements of the selected mask |
| `eta_lower`, `eta_upper`, `oracle_calls` | Ratio bracket and number of Bellman solves |
| `chunks`, `short_chunks`, `shortfall_rows`, `overhead_ms`, `oracle_intervals` | Fragmentation and selected intervals |
| `gap_lower_pct`, `gap_upper_pct` | Bounds on the mask's relative gap to the optimal ratio; undefined for zero importance |

The saved input hashes and `reference_origin` identify how the reference was generated. The analysis notebook reconstructs mask images from `oracle_intervals`. [Reporting identities](#reporting-identities) explain the fragmentation and gap formulas.

In [35]:
def mask_stats(problem: SelectionProblem, mask, R, lower, upper):
    v = problem.v
    cost = problem.cost
    selected = int(mask.sum())
    I = problem.total_importance(mask)
    L = problem.total_latency(mask)
    spans = runs(mask)
    lengths = np.diff(spans, axis=1).ravel()
    shortfall = np.maximum(cost.s - lengths, 0).sum()
    comparable = selected == R and lower > 0
    status = "empty_selection" if selected == 0 else "ok" if v.any() else "zero_importance"
    return dict(
        R_selected=selected, fill_pct=100*selected/R, status=status,
        importance=I, latency=L, retained_pct=100*I/v.sum() if v.any() else np.nan,
        chunks=len(spans), short_chunks=int((lengths < cost.s).sum()),
        shortfall_rows=shortfall, overhead_ms=(cost.b2-cost.b1)*shortfall,
        gap_lower_pct=max(0, 100*(1-I/L/lower)) if comparable else np.nan,
        gap_upper_pct=max(0, 100*(1-I/L/upper)) if comparable else np.nan,
        intervals=json.dumps(spans.tolist()))

### Solve all requested budgets

Evaluate each problem at every requested fraction and collect one record per trace-budget pair.

In [36]:
records = []
progress = tqdm(cases, desc="Oracle reference", unit="trace")
for trace, problem in progress:
    progress.set_postfix(trace_id=trace["trace_id"], refresh=False)
    for fraction in BUDGETS:
        R = max(1, int(fraction * len(problem.v)))
        mask, lower, upper, calls = problem.solve(R)
        stats = mask_stats(problem, mask, R, lower, upper)

        records.append(dict(
            trace_id=trace["trace_id"], shape=trace["shape"],
            prompt=trace["input_index"], fraction=fraction,
            N=len(problem.v), R_nominal=R, saturation_rows=problem.cost.s,
            eta_lower=lower, eta_upper=upper, oracle_calls=calls,
            reference_origin="Dinkelbach DP; rtol=1e-6",
            **input_hashes, **stats,
        ))

Oracle reference: 100%|██████████| 128/128 [01:16<00:00,  1.67trace/s, trace_id=503]


### Inspect the reference

Use the output column names expected by the downstream notebooks and check that each trace-budget pair appears once.

In [37]:
reference = pd.DataFrame(records).rename(columns={
    "importance": "oracle_importance",
    "latency": "oracle_latency",
    "intervals": "oracle_intervals",
})
assert not reference.duplicated(["trace_id", "fraction"]).any()
reference.head()

,trace_id,shape,prompt,fraction,N,R_nominal,saturation_rows,eta_lower,eta_upper,oracle_calls,...,oracle_importance,oracle_latency,retained_pct,chunks,short_chunks,shortfall_rows,overhead_ms,gap_lower_pct,gap_upper_pct,oracle_intervals
0,0,896x896,0,0.1,896,89,118.857143,3883.141069,3883.141069,1,...,121.423778,0.031269,13.551761,1,1,29.857143,0.003070,0,0.000000e+00,"[[483, 572]]"
1,0,896x896,0,0.2,896,179,118.857143,3852.504241,3852.504241,2,...,241.756131,0.062753,26.981711,2,2,58.714286,0.006037,0,2.220446e-14,"[[13, 86], [481, 587]]"
2,0,896x896,0,0.3,896,268,118.857143,3728.043190,3728.043190,2,...,350.519778,0.094022,39.120511,3,3,88.571429,0.009107,0,0.000000e+00,"[[35, 86], [483, 585], [670, 785]]"
3,0,896x896,0,0.4,896,358,118.857143,3748.035029,3748.035029,1,...,425.148908,0.113432,47.449655,3,0,0.000000,0.000000,0,0.000000e+00,"[[8, 127], [483, 603], [670, 789]]"
4,0,896x896,0,0.5,896,448,118.857143,3631.155947,3631.155947,2,...,525.785880,0.144798,58.681460,4,2,27.714286,0.002849,0,0.000000e+00,"[[13, 108], [313, 432], [483, 602], [670, 785]]"


### Save the reference

This cell replaces `oracle_reference.csv`. Regenerate it when the inputs, trace sample, budgets, or solver settings change.

In [38]:
reference.to_csv(RESULTS, index=False)
RESULTS

PosixPath('oracle_reference.csv')

<a id="derivations"></a>
## 7. Detailed derivations

The executable experiment is complete above. These sections justify the solver and the reported quantities:

- [A. Bounds and convergence](#bounds)
- [B. Bellman recurrence](#recurrence)
- [C. Two range maxima per row](#range-maxima)
- [D. Recovery and complexity](#recovery)
- [E. Reporting identities](#reporting-identities)

<a id="bounds"></a>
### A. Bounds and convergence

For the additive optimum $F_R(q)=\max_M[I(M)-qL(M)]$, all feasible latencies are positive. Hence $F_R(q)\ge0$ exactly when $q\le\eta_R^\star$, and $F_R(\eta_R^\star)=0$.

Let $M_C$ be the most important interval of exactly $R$ rows, found by prefix sums. For any mask with $R$ selected rows,

$$L(M)=b_1R+(b_2-b_1)\sum_C\max(s,|C|)
\ge b_1R+(b_2-b_1)\max(s,R)=T(R).$$

The contiguous seed attains this minimum cost. Top-$R$ selects the largest individual values and gives the maximum importance. Hence

$$\ell_0=\frac{I(M_C)}{T(R)}\le\eta_R^\star\le u_0=\frac{I(\mathrm{Top}\text{-}R)}{T(R)}.$$

The lower bound belongs to a feasible mask. The upper bound combines the best numerator and denominator, which may come from different masks.

Use the Dinkelbach method: set $q=\ell$, keep it fixed during one additive solve, recover a maximizing mask $M$, and update the lower bound to $I(M)/L(M)$. In exact arithmetic,

$$\frac{I(M)}{L(M)}=q+\frac{F_R(q)}{L(M)}.$$

Thus $F_R(q)>0$ improves the feasible ratio, while $F_R(q)=0$ proves optimality. Since every feasible latency is at least $T(R)$, the same solve also gives an upper bound:

$$\eta_R^\star\le q+\frac{F_R(q)}{T(R)},\qquad
u\leftarrow\min\left(u,\ q+\frac{F_R(q)}{T(R)}\right).$$

The upper bound follows by writing the optimal ratio as

$$\eta_R^\star=q+\frac{I^\star-qL^\star}{L^\star}
\le q+\frac{F_R(q)}{L^\star}
\le q+\frac{F_R(q)}{T(R)}.$$

The last inequality uses $F_R(q)\ge0$, which holds because $q$ is the current feasible lower bound.

Keep the best feasible mask and stop when $u-\ell\le\epsilon\ell_0$. This gives

$$0\le1-\frac{I(M)/L(M)}{\eta_R^\star}\le\epsilon.$$

For nonzero importance, $\ell_0>0$; zero importance returns immediately. If $q<\eta_R^\star$, then $F_R(q)>0$ and the next feasible ratio strictly increases. There are only $\binom NR$ feasible masks and therefore finitely many feasible ratios. Exact updates must reach the optimum after finitely many steps; a solve there gives zero residual and makes the bounds meet. The implementation can stop earlier when the bracket satisfies the tolerance. Unlike bisection, it does not guarantee that each iteration halves the bracket. With $K$ additive solves, the search costs $O(K(N-R+1)(R+1))$ time after initialization. The implementation evaluates the bounds in float64, clamps a negative residual to zero, keeps the bounds ordered, and raises an error if progress stalls before the requested tolerance is met.

<a id="recurrence"></a>
### B. Bellman recurrence

For a proposed ratio $q\ge0$, define

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

Every feasible latency is positive, so $F_R(q)\ge0$ exactly when $q\le\eta_R^\star$. One additive optimization therefore tests whether a ratio is attainable.

Write $Z=N-R$ and $P_j=\sum_{i<j}v_i$. Let $D_{z,r}$ be the largest $I-qL$ over prefixes of length $z+r$ with $z$ zeros and $r$ ones. With no zeros,

$$D_{0,r}=P_r-qT(r).$$

For $z>0$, place the last zero after a prefix containing $z-1$ zeros and $t$ ones. The final run then occupies $[z+t,z+r)$ and has length $r-t$. Allowing an empty final run gives

$$\boxed{D_{z,r}=\max_{0\le t\le r}\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].}$$

Every feasible prefix has this decomposition. For a fixed $t$, replacing its earlier prefix by the best $D_{z-1,t}$ leaves the final run and its cost unchanged. Conversely, an optimal earlier prefix, one zero and the final run form a feasible prefix. Induction gives $F_R(q)=D_{Z,R}$; the separating zero keeps adjacent pieces from being charged separately.

There are $(Z+1)(R+1)$ states. Scanning all $t$ takes $O((Z+1)(R+1)^2)$ time for a general run-length table. The two-line form reduces this cost.

<a id="range-maxima"></a>
### C. Two range maxima per row

For a fixed $z$, define

$$A_{z,t}=D_{z-1,t}-P_{z+t}+qb_1t,\qquad B_{z,t}=D_{z-1,t}-P_{z+t}+qb_2t.$$

Let $h=\min(\lfloor s\rfloor,R)$; only the integer window is rounded, while $a=(b_2-b_1)s$ stays unchanged. Separating empty, short and long final runs yields

$$D_{z,r}=\max\left\{\begin{aligned}&D_{z-1,r},\\&P_{z+r}-q(a+b_1r)+\max_{\max(0,r-h)\le t<r} A_{z,t},\\&P_{z+r}-qb_2r+\max_{0\le t<r-h} B_{z,t}.\end{aligned}\right.$$

An empty maximum is $-\infty$. The short range is a moving window; the long range is a growing prefix. Each row takes $O(R+1)$ time using a linear-time moving maximum and a prefix maximum.

`trailing_max(values, width)[j]` returns the maximum over indices $\max(0,j-\mathrm{width}+1)$ through $j$. SciPy's `maximum_filter1d` computes these maxima in linear time in compiled code. `origin=(width - 1) // 2` aligns the window with the last `width` values, and constant `-np.inf` padding excludes positions before the array. `np.maximum.accumulate` provides the prefix maximum.

In the code, `A[t]` and `B[t]` are the arrays for the current `z`. `short_max[r]` and `long_max[r]` store the two range maxima above. The final `np.maximum.reduce` compares the empty, short and long final-run scores in the same order as the recurrence.

<a id="recovery"></a>
### D. Recovery and complexity

At $(z,r)$, choose a maximizing $t$ in the original recurrence, select $[z+t,z+r)$ and continue at $(z-1,t)$. At $z=0$, the remaining prefix is all ones. Ties may produce different optimal masks.

Keeping the value table and rescanning one row per recovery step gives

$$O((N-R+1)(R+1))\quad\text{time and memory per additive solve, including recovery}.$$

For $R\simeq N/2$ this is quadratic; the bound also covers $R=N$.

<a id="reporting-identities"></a>
### E. Reporting identities

For selected runs $\mathcal C(M)$, define the total short-run deficit

$$D(M)=\sum_{C\in\mathcal C(M)}(s-|C|)_+.$$

The model can also be written as $T(\ell)=b_2\ell+(b_2-b_1)(s-\ell)_+$ for positive run lengths. Summing over runs gives

$$L(M)=b_2|M|+(b_2-b_1)D(M).$$

These quantities correspond to `shortfall_rows` and `overhead_ms`; `short_chunks` counts runs with length strictly below $s$. The identity also applies to masks selecting fewer than $R$ rows.

#### Comparing ratios at the same budget

For a mask with exactly $R$ rows and ratio $u_M=I(M)/L(M)$, define $g(M)=100(1-u_M/\eta_R^\star)$. The oracle bracket gives

$$\max(0,100(1-u_M/\ell))\le g(M)\le\max(0,100(1-u_M/u)).$$

We report the upper end; the interval width is at most $100\epsilon$ percentage points in exact arithmetic. A mask selecting fewer than $R$ rows is outside the fixed-$R$ feasible set, so its gap is undefined. Comparisons with other selectors are performed in [02_03](02_03_selector_sweep.ipynb). Empty selections and zero-importance cases also have no ratio gap, but retain their other measurements and status.